# Experiment 02b — Enhanced linear vs GBR (mode indicators)

**Paper reference.** Discussion in §2.2 / Appendix B.2 of v15.

**Goal.** The vanilla linear families I–VII are smooth in the inputs and cannot capture two discrete modes: (a) the seller is sitting near the floor and barely moves, and (b) the buyer just retreated. We augment the design matrix with three indicator features and one interaction and check how much of the linear-vs-GBR gap that closes.

In [2]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = 'merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

rows=45,779  policies=['PVD', 'IC', 'FRB', 'RFK', 'OADS', 'MEVJ']  schemes=['A', 'B', 'C', 'D']


In [3]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

Loaded selected_linear.csv:


,scheme,policy,best_family,best_model,best_mae,best_mae$,chosen_family,chosen_model,chosen_dim_phi,chosen_tau,chosen_theta_l1,chosen_mae,chosen_mae$,gap_rel
0,A,FRB,VII,GBR,0.695431,333.803356,VII,GBR,7,2,NaN,0.695431,333.803356,0.000000
1,A,IC,IV,GBR,0.575605,386.213163,IV,GBR,6,1,NaN,0.575605,386.213163,0.000000
2,A,MEVJ,V,GBR,0.405730,447.059539,V,GBR,3,1,NaN,0.405730,447.059539,0.000000
3,A,OADS,VII,GBR,0.664666,375.987772,VII,GBR,7,2,NaN,0.664666,375.987772,0.000000
4,A,PVD,IV,GBR,0.366437,418.193682,V,OLS,3,1,0.722564,0.389880,562.509689,0.063976


## 1. Mode prevalence by (scheme, policy)

Define three observed modes:

- `reject` : S̃ₜ < 0.05  (seller close to floor)
- `persist`: |S̃ₜ − S̃ₜ₋₁| < 1e-3 and not in `reject`
- `concede`: everything else

In [4]:
def assign_mode(d):
    m = pd.Series('concede', index=d.index)
    m[d['S_tilde'] < 0.05] = 'reject'
    m[(np.abs(d['S_tilde'] - d['S_tilde_lag1']) < 1e-3) & (d['S_tilde'] >= 0.05)] = 'persist'
    return m

clean = turns[(~turns.parse_fail) & turns['round'].between(3, 12)
              & turns['S_tilde'].notna() & turns['S_tilde_lag1'].notna()].copy()
clean['mode'] = assign_mode(clean)
prev = (clean.groupby(['scheme','policy'])['mode'].value_counts(normalize=True)
        .unstack(fill_value=0).round(3))
prev

mode           concede  persist  reject
scheme policy                          
A      FRB       0.445    0.014   0.541
       IC        0.404    0.098   0.498
       MEVJ      0.562    0.111   0.327
       OADS      0.467    0.038   0.496
       PVD       0.490    0.204   0.306
       RFK       0.337    0.276   0.387
B      FRB       0.388    0.003   0.609
       IC        0.395    0.030   0.575
       MEVJ      0.587    0.017   0.396
       OADS      0.500    0.013   0.487
       PVD       0.630    0.058   0.313
       RFK       0.421    0.026   0.553
C      FRB       0.547    0.110   0.343
       IC        0.358    0.351   0.291
       MEVJ      0.502    0.357   0.141
       OADS      0.504    0.252   0.244
       PVD       0.426    0.461   0.113
       RFK       0.391    0.406   0.203
D      FRB       0.562    0.060   0.378
       IC        0.574    0.107   0.319
       MEVJ      0.696    0.131   0.174
       OADS      0.660    0.108   0.232
       PVD       0.714    0.150   0.136
       RFK       0.617    0.084   0.298

## 2. Build augmented designs and re-run CV

For each (scheme, policy) we compare four candidate models on the **chosen** family from `selected_linear.csv`:

- **Vanilla linear** (Ridge on the chosen family)
- **Augmented linear** (chosen family + 3 mode indicators + one floor interaction)
- **GBR ceiling** on the same chosen family (no augmentation)

All under grouped 10-fold CV on D_negotiation.

In [5]:
def eval_combo(meta, X, y, model_name):
    pred, _ = nv.grouped_cv_predict(meta, X, y, model_name)
    return nv.cv_metrics(pred), pred

rows = []
oof  = {}
for sch in SCHEMES:
    for pol in POLICIES:
        fam, _ = picked(sch, pol)
        sub = turns.query('scheme == @sch and policy == @pol')
        meta, X, y, cols = nv.build_design_matrix(sub, fam)
        (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)

        # vanilla
        m_v, oof[(sch,pol,'vanilla')] = eval_combo(m_n, X_n, y_n, 'Ridge')
        rows.append({'scheme': sch, 'policy': pol, 'approach': 'vanilla', 'dim': X_n.shape[1], **m_v})

        # augmented (mode indicators)
        X_aug, cols_aug = nv.add_mode_indicators(m_n, X_n, cols)
        m_a, oof[(sch,pol,'augmented')] = eval_combo(m_n, X_aug, y_n, 'Ridge')
        rows.append({'scheme': sch, 'policy': pol, 'approach': 'augmented', 'dim': X_aug.shape[1], **m_a})

        # GBR ceiling
        m_g, oof[(sch,pol,'gbr')] = eval_combo(m_n, X_n, y_n, 'GBR')
        rows.append({'scheme': sch, 'policy': pol, 'approach': 'gbr', 'dim': X_n.shape[1], **m_g})

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUT_DIR, 'exp02b_results.csv'), index=False)
results.head(12).round(4)

KeyboardInterrupt: 

## 3. How much of the linear↔GBR gap does augmentation close?

In [ ]:
piv = (results.pivot_table(index=['scheme','policy'], columns='approach',
                            values='mae').round(4))
piv['gap_vanilla'] = piv['vanilla'] - piv['gbr']
piv['gap_aug']     = piv['augmented'] - piv['gbr']
piv['closed_pct']  = (1 - piv['gap_aug'] / piv['gap_vanilla'].replace(0, np.nan)) * 100
piv.round(2)

In [ ]:
# A boxplot of the relative gap closure across the 24 cells
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot([piv['gap_vanilla'].dropna(), piv['gap_aug'].dropna()],
           labels=['vanilla → GBR', 'augmented → GBR'])
ax.set_ylabel('MAE gap'); ax.axhline(0, color='k', lw=0.5)
ax.set_title('Linear-to-GBR gap shrinks after adding 3 mode indicators')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp02b_gap.png'))
plt.show()

**Reading.** If `closed_pct` is high (say > 50%) for most cells, the residual nonlinearity in §2.2 is mostly the floor/retreat regime, not a deeper problem with the linear-on-logit form. That's a clean defensive point against a reviewer who says 'a smooth linear model can't possibly capture LLM behaviour'.